# 03 — Streaming silver/gold with watermarking, de-dup, and idempotent upsert (EMR)

Consume a bronze table as a stream, produce a watermarked/deduplicated silver stream, enrich with dimensions, and upsert windowed revenue into gold using `foreachBatch` + `MERGE` for idempotent, replay-safe writes.

If you ran `07_file_rate_streaming_fallback.ipynb` instead of `02_kafka_msk_streaming_ingest.ipynb`, set `bronze_table` below to `bronze_clickstream_rate`.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"
bronze_table = "bronze_clickstream_kafka"  # or "bronze_clickstream_rate" if you ran 07 instead of 02

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

if not spark.catalog.tableExists(f"{cfg.schema}.{bronze_table}"):
    raise ValueError(
        f"{bronze_table} does not exist yet. Run 02_kafka_msk_streaming_ingest.ipynb (or the "
        "07_file_rate_streaming_fallback.ipynb fallback and set bronze_table to its output table) first."
    )

## Silver: watermark, de-duplicate, normalize, enrich

In [ ]:
from retail_lakehouse.streaming import deduplicate_stream
from retail_lakehouse.transformations import normalize_click_events, enrich_with_product_customer

bronze_stream = spark.readStream.table(cfg.table(bronze_table))

deduped_stream = deduplicate_stream(bronze_stream, watermark_col="event_ts", watermark_delay="10 minutes")
normalized_stream = normalize_click_events(deduped_stream)

products = spark.table(cfg.table("dim_product_seed"))
customers = spark.table(cfg.table("dim_customer_seed"))
silver_stream = enrich_with_product_customer(normalized_stream, products, customers)

silver_query = (
    silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", cfg.checkpoint("silver_clickstream_streaming"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(cfg.table("silver_clickstream_streaming"))
)
silver_query.awaitTermination()
print("Silver rows:", spark.table(cfg.table("silver_clickstream_streaming")).count())

## Gold: windowed revenue via idempotent MERGE upsert

`make_gold_upsert` (from `retail_lakehouse.streaming`) builds a `foreachBatch` function that MERGEs each micro-batch's aggregated rows by `(window_start, category)`. Replaying a micro-batch after a retry updates the same gold rows instead of double-counting revenue -- this is what "effectively-exactly-once" means in practice for a streaming aggregation sink.

In [ ]:
from pyspark.sql import functions as F
from retail_lakehouse.streaming import windowed_revenue, make_gold_upsert

gold_table = cfg.table("gold_revenue_windows_streaming")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_table} (
  window STRUCT<start: TIMESTAMP, end: TIMESTAMP>,
  category STRING,
  orders LONG,
  revenue DOUBLE
) USING DELTA
LOCATION '{cfg.path("tables", "gold_revenue_windows_streaming")}'
""")

silver_for_gold = (
    spark.readStream.table(cfg.table("silver_clickstream_streaming"))
    .withWatermark("event_ts", "10 minutes")
)
windowed = windowed_revenue(silver_for_gold, window_duration="5 minutes", watermark_delay="10 minutes")

upsert_fn = make_gold_upsert(
    target_table=gold_table,
    merge_keys=["window", "category"],
    update_cols=["orders", "revenue"],
)

gold_query = (
    windowed.writeStream
    .foreachBatch(upsert_fn)
    .option("checkpointLocation", cfg.checkpoint("gold_revenue_windows_streaming"))
    .trigger(availableNow=True)
    .start()
)
gold_query.awaitTermination()

spark.table(gold_table).orderBy(F.desc("window.start"), F.desc("revenue")).limit(20).show(truncate=False)

## Verify idempotency

Re-running the gold query against already-processed offsets should not change row counts or revenue totals, because MERGE keys on `(window, category)`. Try re-running the previous cell now -- row count and revenue sums should be identical.

In [ ]:
before = spark.table(gold_table).agg(F.sum("revenue")).first()[0]
print("Total gold revenue (should stay identical across reruns of the previous cell):", before)

## Next

`04_delta_production_patterns_iceberg.ipynb` covers Delta production operations (MERGE, time travel, OPTIMIZE, VACUUM) and a managed-Iceberg comparison; `05_observability_testing_performance.ipynb` covers monitoring these queries in production.